In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm
from pathlib import Path
from matplotlib.lines import Line2D


In [ ]:
# Combine metrics
UNCONDITIONAL_METRICS_PATH = Path("runs/unconditional.csv")
CONDITIONAL_METRICS_PATH = Path("runs/latent_survival_alpha=1e-1.csv")

NUM_POINTS_PER_PROMPT = 16
num_prompts = 8 # i.e. don't limit
unconditional_metrics_df = pd.read_csv(UNCONDITIONAL_METRICS_PATH)[:NUM_POINTS_PER_PROMPT * num_prompts]
conditional_metrics_df = pd.read_csv(CONDITIONAL_METRICS_PATH)[:NUM_POINTS_PER_PROMPT * num_prompts]

# Select values of prompt_id that are in both dataframes
unconditional_metrics_df = unconditional_metrics_df[unconditional_metrics_df["prompt_id"].isin(conditional_metrics_df["prompt_id"])]
conditional_metrics_df = conditional_metrics_df[conditional_metrics_df["prompt_id"].isin(unconditional_metrics_df["prompt_id"])]

# Combine, with a column indicating the run type
unconditional_metrics_df["run_type"] = "unconditional"
conditional_metrics_df["run_type"] = "latent_survival"
metrics_df = pd.concat([unconditional_metrics_df, conditional_metrics_df])


In [ ]:
metrics_df.loc[(metrics_df["run_type"] == "latent_survival") & (metrics_df["chosen"] == True)]

In [ ]:
def plot_user_score_by_round_and_run_type(metrics_df, agg="max"):
    """
    Plots user_score by round and run_type, aggregating with the specified method.

    Parameters:
    - metrics_df: pd.DataFrame with columns ['run_type', 'prompt_id', 'round', 'user_score']
    - agg: str, one of 'max', 'min', 'mean'
    
    Returns:
    - fig: matplotlib.figure.Figure
    """
    run_types = metrics_df['run_type'].unique()
    prompt_ids = metrics_df['prompt_id'].unique()

    # Assign colors to prompt_ids
    colors = plt.get_cmap('tab10').colors
    if len(prompt_ids) > 10:
        colors = plt.get_cmap('tab20').colors
    color_map = dict(zip(prompt_ids, colors))

    fig, ax = plt.subplots(figsize=(10, 6))

    # Plot each combination of run_type and prompt_id
    for run_type in run_types:
        linestyle = '-' if run_type == "latent_survival" else '--'
        for prompt_id in prompt_ids:
            subset = metrics_df[(metrics_df['run_type'] == run_type) & (metrics_df['prompt_id'] == prompt_id)]
            aggregated_df = subset.groupby('round')['user_score'].agg(agg).reset_index()
            
            # Plot line connecting aggregated points
            ax.plot(aggregated_df['round'], aggregated_df['user_score'],
                    color=color_map[prompt_id], linestyle=linestyle, alpha=0.8)

    # Custom legend for run_type only
    legend_elements = [
        Line2D([0], [0], color='black', linestyle='--', label='unconditional'),
        Line2D([0], [0], color='black', linestyle='-', label='latent_survival')
    ]
    ax.legend(handles=legend_elements)

    NUM_ROUNDS = metrics_df['round'].max() + 1
    ax.set_xticks(np.arange(0, NUM_ROUNDS, 1))
    ax.set_xlabel("Round")
    ax.set_ylabel("User Score (ImageReward)")
    ax.set_title(f"{agg.capitalize()} user score over adaptation rounds (separated by method and prompt)")
    
    return fig


In [ ]:
plot_user_score_by_round_and_run_type(
    metrics_df, 
    "max"
)